# 044 — Detección de anomalías

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Estadístico:** `z = (x−μ)/σ` con |z|>3 como referencia bajo normalidad. Los outliers
inflan σ y se esconden (*masking*): versión robusta `z = (x − mediana)/(1.4826·MAD)`.

**Isolation Forest:** las anomalías son pocas y diferentes → se aíslan con menos cortes
aleatorios. `score(x) = 2^(−E[h(x)]/c(n))`; camino corto ⇒ score → 1. Casi lineal, sin
supuesto de distribución; `contamination` fija el umbral (decisión, no propiedad).

**LOF:** compara densidad local con la de los k vecinos; LOF ≫ 1 = raro *para su región*
(anomalía local que los métodos globales no ven). Costo O(n²), sensible a k.

**Evaluación:** con 0.1-5 % de anomalías la accuracy es inútil; se usa precision/recall de
alarmas y el umbral se fija por capacidad de revisión y costos FN/FP. Outlier (error de
dato) ≠ anomalía de interés: el triage es humano.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (a) μ = 46.44, σ² poblacional = media de (x−μ)² ≈ 5181 → σ ≈ 72.0;
z(250) = (250−46.44)/72.0 ≈ **2.83**: NO supera 3. (b) mediana = 21,
desvíos |x−21| = [1,1,0,2,2,1,1,0,229] → MAD = 1; z_rob = 229/(1.4826·1) ≈ **154.5**.
(c) Solo el robusto detecta: la anomalía contaminó μ y σ (los infló) hasta quedar por
debajo de su propio umbral.

**Ejercicio 2.** Con 260 añadido, μ = 67.8 y σ ≈ 93.6 → z(250) = (250−67.8)/93.6 ≈
**1.95**: BAJA. Cada anomalía adicional esconde mejor a las demás (masking): en series con
múltiples anomalías los estimadores clásicos colapsan y los robustos (breakdown point 50 %)
sobreviven.

**Ejercicio 3.** (a) El rango es 490; la zona (13, 500) mide 487 → probabilidad
487/490 ≈ **0.994** de aislar al 500 con el primer corte: E[h(500)] ≈ 1. (b) El 12 está
rodeado (10-13): los cortes deben caer repetidamente dentro de un intervalo de ancho 3
para separarlo de sus vecinos → E[h(12)] alto. Con la fórmula
`score = 2^(−E[h]/c(n))`: h pequeño ⇒ score cerca de 1 (anomalía); h grande ⇒ score
hacia 0.5 (normal).

**Ejercicio 4.** (a) Revisar el top 50/10 000 = **percentil 99.5** de scores (la capacidad
fija el umbral). (b) Fraudes diarios ≈ 20. El top-1 % (100 casos) contiene ~16 fraudes
(80 %); si el detector ordena bien dentro de ese grupo, las 50 alarmas del top-0.5 %
capturan una parte de esos ~16 — en el mejor caso los 16 (precision 16/50 = 0.32); con
orden aleatorio dentro del top-1 %, ~8 (precision ≈ 0.16). El dato que falta —cómo se
distribuyen los fraudes dentro del top— es exactamente lo que un buen backtesting mediría.


In [ ]:
result = run_lab("ml", seed=44)
assert result["kind"] == "ml"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicios 1 y 2 — z clásico vs. robusto, y masking
import statistics as st

def z_clasico(x, xs):
    mu = st.mean(xs)
    sigma = st.pstdev(xs)
    return (x - mu) / sigma

def z_robusto(x, xs):
    med = st.median(xs)
    mad = st.median([abs(v - med) for v in xs])
    return (x - med) / (1.4826 * mad)

lat = [20, 22, 21, 19, 23, 20, 22, 21, 250]
print(f"z clásico(250) = {z_clasico(250, lat):.2f}   z robusto(250) = {z_robusto(250, lat):.1f}")

lat2 = lat + [260]
print(f"con segunda anomalía: z clásico(250) = {z_clasico(250, lat2):.2f}  ← masking, baja aún más")


In [ ]:
# Ejercicio 4 — umbral operativo
n_diarias, capacidad = 10_000, 50
tasa_fraude, recall_top1pct = 0.002, 0.80

percentil = 100 * (1 - capacidad / n_diarias)
fraudes_dia = n_diarias * tasa_fraude
fraudes_top1pct = fraudes_dia * recall_top1pct
print(f"umbral operativo: percentil {percentil:.1f} del score")
print(f"fraudes/día ≈ {fraudes_dia:.0f}; en el top-1 % (100 casos) ≈ {fraudes_top1pct:.0f}")
print(f"precision en 50 alarmas: mejor caso {min(fraudes_top1pct, 50) / 50:.2f}, "
      f"orden aleatorio dentro del top ≈ {fraudes_top1pct * 0.5 / 50:.2f}")


## Reflexión

1. En el ejemplo trabajado, el fraude de 480 USD produce z ≈ 3.0 con estimadores clásicos
   y z ≈ 210 con mediana/MAD. ¿Qué propiedad de la media y la desviación estándar explica
   la diferencia y cómo se llama el efecto?
2. ¿Por qué `contamination=0.05` no es un hecho sobre los datos sino una decisión
   operativa, y qué información del negocio usarías para fijarla?
3. Da un ejemplo concreto de anomalía que LOF detectaría e Isolation Forest
   probablemente no, y explica por qué con el mecanismo de cada método.
